In [1]:
from __future__ import annotations
import operator
import os
import re
import urllib.request
from datetime import date, timedelta
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from pydantic import BaseModel, Field
from dotenv import load_dotenv, find_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

load_dotenv(find_dotenv(usecwd=True) or '.env', override=True)


<cell>:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.


True

In [2]:
class Task(BaseModel):
    id: int
    title: str

    goal: str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section",
    )
    bullets: List[str] = Field(
        ...,
        min_length=3,
        max_length=6,
        description="3-6 concrete, non-overlapping subpoints to cover in this section.",
    )
    target_words: int = Field(
        ...,
        description="Target word count for this section (120-550)"
    )
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citation: bool = False
    requires_citations: bool = False
    requires_code: bool = False

class Plan(BaseModel):
    blog_title: str
    audience: str
    tone: str
    blog_kind: Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constraints: List[str] = Field(default_factory=list)
    constrains: List[str] = Field(default_factory=list)
    tasks: List[Task]

class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None
    snippet: Optional[str] = None
    source: Optional[str] = None

class RouterDecision(BaseModel):
    needs_research: bool
    mode: Literal["closed_book", "hybrid", "open_book"]
    queries: List[str] = Field(default_factory=list)

class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)

class ImageSpec(BaseModel):
    placeholder: str = Field(
        ...,
        description="e.g. [[IMAGE_1]]"
    )
    filename: str = Field(
        ...,
        description="Save under images/, e.g. images/architecture.png"
    )
    alt: str
    caption: str
    prompt: str = Field(
        ...,
        description="Prompt to send to the image model."
    )
    size: Literal["1024x1024", "1024x1536", "1536x1024"] = "1024x1024"
    quality: Literal["low", "medium", "high"] = "medium"

class GlobalImagePlan(BaseModel):
    md_with_placeholder: str
    images: List[ImageSpec] = Field(default_factory=list)

In [3]:
class State(TypedDict, total=False):
    topic: str
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    sections: Annotated[List[tuple[int, str]], operator.add]

    merged_md: str
    md_with_placeholders: str
    image_specs: List[dict]

    final: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
openai_client = OpenAI()

In [4]:
ROUTER_SYSTEM = """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""

def router_node(state: State) -> dict:
    topic = state["topic"]
    decider = llm.with_structured_output(RouterDecision)
    decision = decider.invoke([
        SystemMessage(content=ROUTER_SYSTEM),
        HumanMessage(content=f"Topic: {topic}"),
    ])
    return {
        "needs_research": decision.needs_research,
        "mode": decision.mode,
        "queries": decision.queries,
    }

def route_next(state: State) -> str:
    return "research" if state.get("needs_research", False) else "orchestrator"

In [5]:
def _tavily_search(query: str, max_results: int = 5) -> List[dict]:
    try:
        tool = TavilySearchResults(max_results=max_results)
        results = tool.invoke({"query": query})

        normalized: List[dict] = []
        for r in results or []:
            normalized.append(
                {
                    "title": r.get("title") or "",
                    "url": r.get("url") or "",
                    "snippet": r.get("content") or r.get("snippet") or "",
                    "published_at": r.get("published_date") or r.get("published_at"),
                    "source": r.get("source") or "",
                }
            )
        return normalized
    except Exception as e:
        print(f"Warning: Tavily search unavailable or error for query '{query}': {e}")
        return []

RESEARCH_SYSTEM = """Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""

def research_node(state: State) -> dict:
    queries = state.get("queries", []) or []
    max_results = 6

    raw_results: List[dict] = []
    for q in queries:
        raw_results.extend(_tavily_search(q, max_results=max_results))

    if not raw_results:
        return {"evidence": []}

    extractor = llm.with_structured_output(EvidencePack)
    pack = extractor.invoke([
        SystemMessage(content=RESEARCH_SYSTEM),
        HumanMessage(content=f"Raw RESULTS:\n{raw_results}"),
    ])
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e
    return {"evidence": list(dedup.values())}

In [6]:
ORCH_SYSTEM = """You are a senior technical writer and developer advocate.
Your job is to produce a highly actionable outline for a technical blog post.

Hard requirements:
- Create 5–9 sections (tasks) suitable for the topic and audience.
- Each task must include:
  1) goal (1 sentence)
  2) 3–6 bullets that are concrete, specific, and non-overlapping
  3) target word count (120–550)

Quality bar:
- Assume the reader is a developer; use correct terminology.
- Bullets must be actionable: build/compare/measure/verify/debug.
- Ensure the overall plan includes at least 2 of these somewhere:
  * minimal code sketch / MWE (set requires_code=True for that section)
  * edge cases / failure modes
  * performance/cost considerations
  * security/privacy considerations (if relevant)
  * debugging/observability tips

Grounding rules:
- Mode closed_book: keep it evergreen; do not depend on evidence.
- Mode hybrid:
  - Use evidence for up-to-date examples (models/tools/releases) in bullets.
  - Mark sections using fresh info as requires_research=True and requires_citations=True.
- Mode open_book:
  - Set blog_kind = "news_roundup".
  - Every section is about summarizing events + implications.
  - DO NOT include tutorial/how-to sections unless user explicitly asked for that.
  - If evidence is empty or insufficient, create a plan that transparently says "insufficient sources"
    and includes only what can be supported.

Output must strictly match the Plan schema.
"""

def orchestrator_node(state: State) -> dict:
    planner = llm.with_structured_output(Plan)

    evidence = state.get("evidence", []) or []
    mode = state.get("mode", "closed_book")
    plan = planner.invoke(
        [
            SystemMessage(content=ORCH_SYSTEM),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n"
                    f"Mode: {mode}\n\n"
                    f"Evidence (ONLY use for fresh claims; may be empty):\n"
                    f"{[e.model_dump() if hasattr(e, 'model_dump') else e for e in evidence][:16]}"
                )
            ),
        ]
    )
    return {"plan": plan}

def fanout(state: State) -> List[Send]:
    plan = state["plan"]
    evidence = state.get("evidence", []) or []
    return [
        Send(
            "worker",
            {
                "task": task.model_dump() if hasattr(task, "model_dump") else task,
                "topic": state["topic"],
                "mode": state.get("mode", "closed_book"),
                "plan": plan.model_dump() if hasattr(plan, "model_dump") else plan,
                "evidence": [e.model_dump() if hasattr(e, "model_dump") else e for e in evidence],
            },
        )
        for task in plan.tasks
    ]

In [7]:
WORKER_SYSTEM = """You are a senior technical writer and developer advocate.
Write ONE section of a technical blog post in Markdown.

Hard constraints:
- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).
- Start with a '## <Section Title>' heading.

Scope guard:
- If blog_kind == "news_roundup": do NOT turn this into a tutorial/how-to guide.
  Do NOT teach web scraping, RSS, automation, or "how to fetch news" unless bullets explicitly ask for it.
  Focus on summarizing events and implications.

Grounding policy:
- If mode == open_book:
  - Do NOT introduce any specific event/company/model/funding/policy claim unless it is supported by provided Evidence URLs.
  - For each event claim, attach a source as a Markdown link: ([Source](URL)).
  - Only use URLs provided in Evidence. If not supported, write: "Not found in provided sources."
- If requires_citations == true:
  - For outside-world claims, cite Evidence URLs the same way.
- Evergreen reasoning is OK without citations unless requires_citations is true.

Code:
- If requires_code == true, include at least one minimal, correct code snippet relevant to the bullets.

Style:
- Short paragraphs, bullets where helpful, code fences for code.
- Avoid fluff/marketing. Be precise and implementation-oriented.
"""

def worker_node(payload: dict) -> dict:
    task = Task(**payload["task"]) if isinstance(payload["task"], dict) else payload["task"]
    plan = Plan(**payload["plan"]) if isinstance(payload["plan"], dict) else payload["plan"]
    evidence = [EvidenceItem(**e) if isinstance(e, dict) else e for e in payload.get("evidence", [])]
    topic = payload["topic"]
    mode = payload.get("mode", "closed_book")

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    evidence_text = ""
    if evidence:
        evidence_text = "\n".join(
            f"- {e.title} | {e.url} | {e.published_at or 'date:unknown'}".strip()
            for e in evidence[:20]
        )

    section_md = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {getattr(plan, 'constraints', getattr(plan, 'constrains', []))}\n"
                    f"Topic: {topic}\n"
                    f"Mode: {mode}\n\n"
                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: {task.requires_research}\n"
                    f"requires_citations: {task.requires_citations or task.requires_citation}\n"
                    f"requires_code: {task.requires_code}\n"
                    f"Bullets:{bullets_text}\n\n"
                    f"Evidence (ONLY use these URLs when citing):\n{evidence_text}\n"
                )
            ),
        ]
    ).content.strip()

    return {"sections": [(task.id, section_md)]}

In [8]:
def merge_content(state: State) -> dict:
    plan = state["plan"]
    ordered_sections = [md for _, md in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered_sections).strip()
    merged_md = f"# {plan.blog_title}\n\n{body}\n"
    return {"merged_md": merged_md}

IMAGE_SYSTEM = """You are an expert technical editor and visual content designer.
Your task is to take a complete technical blog post in Markdown and identify 1-3 optimal locations for visual diagrams, architectural schemas, or explanatory charts.

Rules:
- Insert placeholders in the markdown text exactly in the format: [[IMAGE_1]], [[IMAGE_2]], etc.
- For each placeholder, create an ImageSpec:
  - placeholder: matching string like "[[IMAGE_1]]"
  - filename: clean filename under images/, e.g., "images/agent_architecture.png"
  - alt: short descriptive alt text
  - caption: informative caption explaining the diagram
  - prompt: detailed prompt suitable for generating an educational diagram/flowchart
  - size: one of "1024x1024", "1024x1536", "1536x1024"
  - quality: "low", "medium", or "high"
- Do NOT alter any existing text, code blocks, or headings other than inserting the placeholder tokens between paragraphs where diagrams add high value.

Return strictly matching the GlobalImagePlan schema.
"""

def decide_images(state: State) -> dict:
    merged_md = state.get("merged_md", "")
    planner = llm.with_structured_output(GlobalImagePlan)
    image_plan = planner.invoke([
        SystemMessage(content=IMAGE_SYSTEM),
        HumanMessage(content=f"Draft Markdown:\n\n{merged_md}"),
    ])
    return {
        "md_with_placeholders": image_plan.md_with_placeholder,
        "image_specs": [img.model_dump() if hasattr(img, "model_dump") else img for img in image_plan.images],
    }

def generate_and_place_images(state: State) -> dict:
    content = state.get("md_with_placeholders") or state.get("merged_md", "")
    image_specs = state.get("image_specs", []) or []
    plan = state.get("plan")

    final_md = content
    for img in image_specs:
        placeholder = img.get("placeholder") if isinstance(img, dict) else img.placeholder
        filename = img.get("filename") if isinstance(img, dict) else img.filename
        caption = img.get("caption") if isinstance(img, dict) else img.caption
        alt = img.get("alt") if isinstance(img, dict) else img.alt
        prompt = img.get("prompt") if isinstance(img, dict) else img.prompt

        # Generate image using OpenAI DALL-E if prompt is present
        if prompt:
            try:
                target_path = Path(filename)
                target_path.parent.mkdir(parents=True, exist_ok=True)
                if not target_path.exists():
                    print(f"Generating image with DALL-E for: {filename}...")
                    res = openai_client.images.generate(
                        model="dall-e-3",
                        prompt=prompt,
                        size="1024x1024",
                        n=1
                    )
                    img_url = res.data[0].url
                    urllib.request.urlretrieve(img_url, str(target_path))
                    print(f"Saved image to {target_path}")
            except Exception as e:
                print(f"Notice: Image generation for {filename} skipped or failed: {e}")

        if Path(filename).exists():
            md_img = f"\n\n![{alt}]({filename})\n*{caption}*\n\n"
        else:
            md_img = f"\n\n> **[Diagram: {alt}]** *{caption}*\n\n"
        final_md = final_md.replace(placeholder, md_img)

    # Save to disk
    blog_title = plan.blog_title if hasattr(plan, 'blog_title') else (plan.get('blog_title', 'technical_blog') if isinstance(plan, dict) else 'technical_blog')
    safe_title = re.sub(r'[^a-zA-Z0-9_\- ]', '', blog_title).strip().lower().replace(" ", "_")
    filename = f"{safe_title}.md"
    Path(filename).write_text(final_md, encoding="utf-8")

    return {"final": final_md}


In [9]:
reducer_graph = StateGraph(State)
reducer_graph.add_node("merge_content", merge_content)
reducer_graph.add_node("decide_images", decide_images)
reducer_graph.add_node("generate_and_place_images", generate_and_place_images)

reducer_graph.add_edge(START, "merge_content")
reducer_graph.add_edge("merge_content", "decide_images")
reducer_graph.add_edge("decide_images", "generate_and_place_images")
reducer_graph.add_edge("generate_and_place_images", END)

reducer_subgraph = reducer_graph.compile()
reducer_subgraph

In [10]:
g = StateGraph(State)
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_subgraph)

g.add_edge(START, "router")
g.add_conditional_edges("router", route_next, {"research": "research", "orchestrator": "orchestrator"})
g.add_edge("research", "orchestrator")

g.add_conditional_edges("orchestrator", fanout, ["worker"])
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

app = g.compile()
app

In [11]:
initial_state = {
    "topic": "Building Autonomous AI Agents with LangGraph: State, Memory, and Multi-Agent Workflows",
    "sections": []
}

result = app.invoke(initial_state)
print(f"Generated Blog Post Title: {result['plan'].blog_title}")
print(f"Total Sections: {len(result['plan'].tasks)}")
print("\n--- Blog Markdown Content Preview ---\n")
print(result["final"][:1000] + "\n...")

  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, 

<cell>:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
